In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF']     = 'expandable_segments:True'

!pip install --no-cache-dir \
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' \
    pyyaml -q

print('✅ Ready')

In [ ]:
import yaml, pathlib, torch, textwrap
from unsloth         import FastLanguageModel
from google.colab    import userdata
from huggingface_hub import login


def load_demo(config_path='config.yaml'):
    """
    Reads config.yaml, authenticates with HuggingFace,
    and loads the trained LoRA adapter from the Hub.
    Returns (model, tokenizer, cfg) ready for inference.
    """
    p = pathlib.Path(config_path)
    if not p.exists():
        raise FileNotFoundError(f"Upload '{config_path}' via the file panel first.")
    cfg = yaml.safe_load(p.read_text())

    token = userdata.get(cfg['export'].get('hub_token_env', 'HF_TOKEN'))
    os.environ['HF_TOKEN'] = token
    login(token=token, add_to_git_credential=False)

    hub_id    = cfg['export']['hub_model_id']
    model_cfg = cfg['model']
    print(f'Loading model from Hub: {hub_id} ...')

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = hub_id,
        max_seq_length = model_cfg['max_seq_length'],
        dtype          = model_cfg['dtype'],
        load_in_4bit   = model_cfg['load_in_4bit'],
    )
    FastLanguageModel.for_inference(model)

    print(f'✅ Model ready — {hub_id}')
    return model, tokenizer, cfg


model, tokenizer, cfg = load_demo()

In [ ]:
# ── Terms shown during presentation ───────────────────────────
SHOWCASE_TERMS = [
    'Myocardial infarction',
    'Pulmonary embolism',
    'Sepsis',
    'Hyperglycemia',
    'Atrial fibrillation',
]


# ── Core inference function ────────────────────────────────────
def demo(term, instruction=None):
    if instruction is None:
        instruction = cfg['dataset']['instruction_variants'][0]

    prompt = (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{term}\n\n"
        f"### Response:\n"
    )

    inputs    = tokenizer(prompt, return_tensors='pt', truncation=True).to('cuda')
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            eos_token_id   = tokenizer.eos_token_id,
            pad_token_id   = tokenizer.eos_token_id,
            use_cache      = True,
            **cfg['inference'],
        )

    response = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    ).strip()

    width = 72
    print('\n' + '━' * width)
    print(f'  Term   : {term}')
    print('━' * width)
    for line in response.replace('. ', '.\n').split('\n'):
        for chunk in textwrap.wrap(line.strip(), width=width - 4) or ['']:
            print(f'  {chunk}')
    print('━' * width)

    return response


# ── Main — runs all showcase terms ────────────────────────────
def main():
    print(f'\n  Medical Term Simplifier — Live Demo')
    print(f'  Model : {cfg["export"]["hub_model_id"]}')
    print(f'  Terms : {len(SHOWCASE_TERMS)}\n')

    for i, term in enumerate(SHOWCASE_TERMS, 1):
        print(f'  [{i}/{len(SHOWCASE_TERMS)}]', end='')
        demo(term)

    print('\n  ✅  Demo complete.')


# ── Run ───────────────────────────────────────────────────────
main()


# ── After main(), call demo() on any term on the fly ──────────
